# Weather Feature Engineering

## Objective

Prepare historical INMET weather data and create weather features
using observations from before purchase.

## Inputs

Annual INMET archives for 2016, 2017, and 2018.

## Rules

- Preserve the original archives.
- Inspect station metadata, timestamps, units, and missing codes.
- Do not interpret missing rainfall as zero.
- Verify time conventions before daily aggregation.

In [3]:
# Purpose: Locate the project and prepare weather data paths.

from pathlib import Path
from zipfile import ZipFile

import pandas as pd
from IPython.display import display

current_dir = Path.cwd()

if (current_dir / "data/processed/research_data.csv").exists():
    PROJECT_ROOT = current_dir
elif (current_dir.parent / "data/processed/research_data.csv").exists():
    PROJECT_ROOT = current_dir.parent
else:
    raise FileNotFoundError("Cannot locate the project data folder.")
WEATHER_RAW_DIR = PROJECT_ROOT / "data/raw/inmet"
WEATHER_OUTPUT_DIR = PROJECT_ROOT / "data/processed"

WEATHER_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Weather input:", WEATHER_RAW_DIR)
print("Weather output:", WEATHER_OUTPUT_DIR)

Weather input: c:\Users\LENOVO\Downloads\Group5_STAT3013.R11-main (1)\Group5_STAT3013.R11-main\data\raw\inmet
Weather output: c:\Users\LENOVO\Downloads\Group5_STAT3013.R11-main (1)\Group5_STAT3013.R11-main\data\processed


In [4]:
# Purpose: Verify the annual archives and count station CSV files.

inventory = []

for year in [2016, 2017, 2018]:
    archive_path = WEATHER_RAW_DIR / f"{year}.zip"

    if not archive_path.exists():
        raise FileNotFoundError(f"Missing archive: {archive_path}")

    with ZipFile(archive_path) as archive:
        csv_names = [
            name for name in archive.namelist()
            if name.lower().endswith(".csv")
            and "__MACOSX" not in name.split("/")
        ]

        if not csv_names:
            raise ValueError(f"No CSV files in {archive_path.name}")

    inventory.append({
        "year": year,
        "archive": archive_path.name,
        "size_mb": round(archive_path.stat().st_size / 1024**2, 2),
        "csv_files": len(csv_names),
    })

display(pd.DataFrame(inventory))

,year,archive,size_mb,csv_files
0,2016,2016.zip,97.69,529
1,2017,2017.zip,104.60,563
2,2018,2018.zip,112.44,596


In [5]:
# Purpose: Inspect station metadata, headers, and sample observations.

for year in [2016, 2017, 2018]:
    archive_path = WEATHER_RAW_DIR / f"{year}.zip"

    with ZipFile(archive_path) as archive:
        csv_names = sorted(
            name for name in archive.namelist()
            if name.lower().endswith(".csv")
            and "__MACOSX" not in name.split("/")
        )

        if not csv_names:
            raise ValueError(f"No CSV files in {archive_path.name}")

        sample_name = csv_names[0]
        raw_bytes = archive.read(sample_name)

    for encoding in ["utf-8-sig", "cp1252", "latin-1"]:
        try:
            sample_text = raw_bytes.decode(encoding)
            break
        except UnicodeDecodeError:
            continue

    print(f"\nYear: {year}")
    print("Sample file:", sample_name)
    print("Encoding used:", encoding)

    for line_number, line in enumerate(
        sample_text.splitlines()[:16], start=1
    ):
        print(f"{line_number:02d}: {line}")


Year: 2016
Sample file: 2016/INMET_CO_DF_A001_BRASILIA_01-01-2016_A_31-12-2016.CSV
Encoding used: cp1252
01: REGIÃO:;CO
02: UF:;DF
03: ESTAÇÃO:;BRASILIA
04: CODIGO (WMO):;A001
05: LATITUDE:;-15,78944444
06: LONGITUDE:;-47,92583332
07: ALTITUDE:;1159,54
08: DATA DE FUNDAÇÃO (YYYY-MM-DD):;2000-05-07
09: DATA (YYYY-MM-DD);HORA (UTC);PRECIPITAÇÃO TOTAL, HORÁRIO (mm);PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB);PRESSÃO ATMOSFERICA MAX.NA HORA ANT. (AUT) (mB);PRESSÃO ATMOSFERICA MIN. NA HORA ANT. (AUT) (mB);RADIACAO GLOBAL (KJ/m²);TEMPERATURA DO AR - BULBO SECO, HORARIA (°C);TEMPERATURA DO PONTO DE ORVALHO (°C);TEMPERATURA MÁXIMA NA HORA ANT. (AUT) (°C);TEMPERATURA MÍNIMA NA HORA ANT. (AUT) (°C);TEMPERATURA ORVALHO MAX. NA HORA ANT. (AUT) (°C);TEMPERATURA ORVALHO MIN. NA HORA ANT. (AUT) (°C);UMIDADE REL. MAX. NA HORA ANT. (AUT) (%);UMIDADE REL. MIN. NA HORA ANT. (AUT) (%);UMIDADE RELATIVA DO AR, HORARIA (%);VENTO, DIREÇÃO HORARIA (gr) (° (gr));VENTO, RAJADA MAXIMA (m/s);VENTO, VELO

In [6]:
# Purpose: Read station metadata and hourly rainfall from one INMET CSV.

from io import StringIO
import unicodedata


def normalize_text(value):
    text = unicodedata.normalize("NFKD", str(value))
    return (
        text.encode("ascii", "ignore")
        .decode()
        .strip()
        .upper()
    )


def read_station_rainfall(raw_bytes):
    # Decode the source without silently replacing characters.
    for encoding in ["utf-8-sig", "cp1252", "latin-1"]:
        try:
            text = raw_bytes.decode(encoding)
            break
        except UnicodeDecodeError:
            continue

    lines = text.splitlines()

    # Locate the data header instead of using a fixed skiprows value.
    header_index = next(
        (
            i for i, line in enumerate(lines)
            if normalize_text(line).startswith("DATA")
            and "HORA" in normalize_text(line)
            and "PRECIPITACAO" in normalize_text(line)
        ),
        None,
    )

    if header_index is None:
        raise ValueError("Cannot identify the weather table header.")

    metadata = {}

    for line in lines[:header_index]:
        parts = line.split(";", 1)

        if len(parts) == 2:
            key = normalize_text(parts[0]).rstrip(":").strip()
            metadata[key] = parts[1].strip()

    state = metadata.get("UF")
    station_id = metadata.get("CODIGO (WMO)")

    if not state or not station_id:
        raise ValueError("Missing station ID or state metadata.")

    raw = pd.read_csv(
        StringIO("\n".join(lines[header_index:])),
        sep=";",
        dtype="string",
    )

    def find_column(prefix):
        matches = [
            column for column in raw.columns
            if normalize_text(column).startswith(prefix)
        ]

        if len(matches) != 1:
            raise ValueError(
                f"Expected one {prefix} column, found: {matches}"
            )

        return matches[0]

    date_column = find_column("DATA")
    hour_column = find_column("HORA")
    rain_column = find_column("PRECIPITACAO")

    # Accept common date and hour formats found in annual archives.
    date_text = raw[date_column].str.strip().str.replace(
        "/", "-", regex=False
    )

    hour_text = (
        raw[hour_column]
        .str.upper()
        .str.replace("UTC", "", regex=False)
        .str.strip()
        .str.replace(":", "", regex=False)
        .str.zfill(4)
    )

    timestamp = pd.to_datetime(
        date_text + " " + hour_text,
        format="%Y-%m-%d %H%M",
        errors="coerce",
        utc=True,
    )

    if timestamp.isna().any():
        raise ValueError(
            f"Invalid timestamps: {timestamp.isna().sum()}"
        )

    rain_text = (
        raw[rain_column]
        .str.strip()
        .str.replace(",", ".", regex=False)
    )

    rainfall = pd.to_numeric(rain_text, errors="coerce")

    invalid_numeric = (
        rain_text.notna()
        & rain_text.ne("")
        & rainfall.isna()
    )

    if invalid_numeric.any():
        raise ValueError(
            "Unrecognized rainfall values: "
            f"{rain_text[invalid_numeric].unique().tolist()[:10]}"
        )

    # Missing sentinel and other negative values are not zero rainfall.
    sentinel_missing = rainfall.eq(-9999)
    other_negative = rainfall.lt(0) & ~sentinel_missing

    rainfall = rainfall.mask(sentinel_missing | other_negative)

    hourly = pd.DataFrame({
        "station_id": station_id,
        "state": state.upper(),
        "timestamp_utc": timestamp,
        "rainfall_mm": rainfall,
    })

    audit = {
        "station_id": station_id,
        "state": state.upper(),
        "encoding": encoding,
        "rows": len(hourly),
        "sentinel_missing_rows": int(sentinel_missing.sum()),
        "other_negative_rows": int(other_negative.sum()),
        "missing_rainfall_rows": int(rainfall.isna().sum()),
        "duplicate_timestamp_rows": int(
            hourly["timestamp_utc"].duplicated(keep=False).sum()
        ),
    }

    return hourly, audit

In [7]:
# Purpose: Test the rainfall reader on one station file per year.

sample_checks = []

for year in [2016, 2017, 2018]:
    archive_path = WEATHER_RAW_DIR / f"{year}.zip"

    with ZipFile(archive_path) as archive:
        csv_names = sorted(
            name for name in archive.namelist()
            if name.lower().endswith(".csv")
            and "__MACOSX" not in name.split("/")
        )

        sample_name = csv_names[0]

        sample_hourly, audit = read_station_rainfall(
            archive.read(sample_name)
        )

    audit["year"] = year
    sample_checks.append(audit)

    print(f"\nYear: {year}")
    print("File:", sample_name)
    display(sample_hourly.head(3))

display(pd.DataFrame(sample_checks))


Year: 2016
File: 2016/INMET_CO_DF_A001_BRASILIA_01-01-2016_A_31-12-2016.CSV


,station_id,state,timestamp_utc,rainfall_mm
0,A001,DF,2016-01-01 00:00:00+00:00,0.0
1,A001,DF,2016-01-01 01:00:00+00:00,0.0
2,A001,DF,2016-01-01 02:00:00+00:00,0.0



Year: 2017
File: 2017/INMET_CO_DF_A001_BRASILIA_01-01-2017_A_31-12-2017.CSV


,station_id,state,timestamp_utc,rainfall_mm
0,A001,DF,2017-01-01 00:00:00+00:00,0.0
1,A001,DF,2017-01-01 01:00:00+00:00,0.0
2,A001,DF,2017-01-01 02:00:00+00:00,0.0



Year: 2018
File: 2018/INMET_CO_DF_A001_BRASILIA_01-01-2018_A_31-12-2018.CSV


,station_id,state,timestamp_utc,rainfall_mm
0,A001,DF,2018-01-01 00:00:00+00:00,0.0
1,A001,DF,2018-01-01 01:00:00+00:00,0.0
2,A001,DF,2018-01-01 02:00:00+00:00,0.0


,station_id,state,encoding,rows,sentinel_missing_rows,other_negative_rows,missing_rainfall_rows,duplicate_timestamp_rows,year
0,A001,DF,cp1252,8784,1,0,1,0,2016
1,A001,DF,cp1252,8760,0,0,0,0,2017
2,A001,DF,cp1252,8760,48,0,48,0,2018


## Station Daily Rainfall

Daily aggregation uses UTC dates.

A daily rainfall total requires 24 distinct hourly timestamps
and 24 valid rainfall observations. Incomplete days remain missing.

Exact duplicate observations are removed. Conflicting observations
for the same station and timestamp require manual review.

In [8]:
# Purpose: Aggregate one station's hourly rainfall into complete UTC days.

def aggregate_station_daily(hourly):
    hourly = hourly.copy()

    if hourly["station_id"].nunique() != 1:
        raise ValueError("Expected exactly one station.")

    if hourly["state"].nunique() != 1:
        raise ValueError("Expected exactly one state.")

    # Remove identical observations only.
    hourly = hourly.drop_duplicates(
        subset=["station_id", "state", "timestamp_utc", "rainfall_mm"]
    )

    if hourly["timestamp_utc"].duplicated().any():
        raise ValueError(
            "Conflicting observations at the same timestamp."
        )

    # Require observations to fall exactly on hourly boundaries.
    if not hourly["timestamp_utc"].eq(
        hourly["timestamp_utc"].dt.floor("h")
    ).all():
        raise ValueError("Non-hourly timestamps detected.")

    station_id = hourly["station_id"].iloc[0]
    state = hourly["state"].iloc[0]

    hourly["weather_date"] = (
        hourly["timestamp_utc"]
        .dt.floor("D")
        .dt.tz_localize(None)
    )

    daily = (
        hourly.groupby("weather_date")
        .agg(
            observed_hours=("timestamp_utc", "nunique"),
            valid_rain_hours=("rainfall_mm", "count"),
            rainfall_mm=(
                "rainfall_mm",
                lambda values: values.sum(min_count=1),
            ),
        )
    )

    # Expose dates with no observations between the first and last date.
    date_index = pd.date_range(
        daily.index.min(),
        daily.index.max(),
        freq="D",
        name="weather_date",
    )

    daily = daily.reindex(date_index)

    for column in ["observed_hours", "valid_rain_hours"]:
        daily[column] = daily[column].fillna(0).astype("int64")

    daily["complete_day"] = (
        daily["observed_hours"].eq(24)
        & daily["valid_rain_hours"].eq(24)
    )

    daily["rainfall_mm"] = daily["rainfall_mm"].where(
        daily["complete_day"]
    )

    daily = daily.reset_index()
    daily.insert(0, "state", state)
    daily.insert(0, "station_id", station_id)

    return daily

In [9]:
# Purpose: Process all station files and collect daily data and audits.

daily_parts = []
file_audits = []

for year in [2016, 2017, 2018]:
    archive_path = WEATHER_RAW_DIR / f"{year}.zip"

    with ZipFile(archive_path) as archive:
        csv_names = sorted(
            name for name in archive.namelist()
            if name.lower().endswith(".csv")
            and "__MACOSX" not in name.split("/")
        )

        for number, name in enumerate(csv_names, start=1):
            try:
                hourly, audit = read_station_rainfall(
                    archive.read(name)
                )

                if not hourly["timestamp_utc"].dt.year.eq(year).all():
                    raise ValueError(
                        "Observation year differs from archive year."
                    )

                daily = aggregate_station_daily(hourly)

            except Exception as error:
                raise RuntimeError(
                    f"Failed to process {year}: {name}"
                ) from error

            audit.update({
                "year": year,
                "source_file": name,
                "daily_rows": len(daily),
                "complete_days": int(daily["complete_day"].sum()),
                "incomplete_days": int((~daily["complete_day"]).sum()),
            })

            daily_parts.append(daily)
            file_audits.append(audit)

            if number % 100 == 0 or number == len(csv_names):
                print(f"{year}: processed {number}/{len(csv_names)} files")

station_daily = pd.concat(daily_parts, ignore_index=True)
weather_file_audit = pd.DataFrame(file_audits)

# Detect overlapping station records across source files.
duplicate_days = station_daily.duplicated(
    ["station_id", "weather_date"],
    keep=False,
)

if duplicate_days.any():
    display(
        station_daily.loc[duplicate_days]
        .sort_values(["station_id", "weather_date"])
        .head(20)
    )
    raise ValueError(
        "Overlapping station dates found. Review before continuing."
    )

display(
    weather_file_audit.groupby("year")
    .agg(
        source_files=("source_file", "size"),
        hourly_rows=("rows", "sum"),
        daily_rows=("daily_rows", "sum"),
        complete_days=("complete_days", "sum"),
        incomplete_days=("incomplete_days", "sum"),
    )
)

print("Unique stations:", station_daily["station_id"].nunique())
print("States represented:", station_daily["state"].nunique())

display(station_daily.head())

2016: processed 100/529 files
2016: processed 200/529 files
2016: processed 300/529 files
2016: processed 400/529 files
2016: processed 500/529 files
2016: processed 529/529 files
2017: processed 100/563 files
2017: processed 200/563 files
2017: processed 300/563 files
2017: processed 400/563 files
2017: processed 500/563 files
2017: processed 563/563 files
2018: processed 100/596 files
2018: processed 200/596 files
2018: processed 300/596 files
2018: processed 400/596 files
2018: processed 500/596 files
2018: processed 596/596 files


,source_files,hourly_rows,daily_rows,complete_days,incomplete_days
year,,,,,
2016,529,4372128,182172,150343,31829
2017,563,4775544,198981,158418,40563
2018,596,5128416,213684,173726,39958


Unique stations: 597
States represented: 27


,station_id,state,weather_date,observed_hours,valid_rain_hours,rainfall_mm,complete_day
0,A001,DF,2016-01-01,24,24,1.8,True
1,A001,DF,2016-01-02,24,24,3.6,True
2,A001,DF,2016-01-03,24,24,32.6,True
3,A001,DF,2016-01-04,24,24,0.2,True
4,A001,DF,2016-01-05,24,24,6.4,True


In [10]:
# Purpose: Aggregate complete station-day rainfall into state-day data.

state_daily = (
    station_daily.groupby(["state", "weather_date"], as_index=False)
    .agg(
        rainfall_mm=("rainfall_mm", "mean"),
        n_valid_stations=("rainfall_mm", "count"),
        n_station_records=("station_id", "nunique"),
    )
)

# Include every state and calendar date, even when observations are absent.
states = sorted(station_daily["state"].unique())

all_dates = pd.date_range(
    "2016-01-01",
    "2018-12-31",
    freq="D",
)

full_index = pd.MultiIndex.from_product(
    [states, all_dates],
    names=["state", "weather_date"],
)

state_daily = (
    state_daily
    .set_index(["state", "weather_date"])
    .reindex(full_index)
    .reset_index()
)

for column in ["n_valid_stations", "n_station_records"]:
    state_daily[column] = (
        state_daily[column].fillna(0).astype("int64")
    )

# This ratio describes available station records, not all active stations.
state_daily["station_record_completeness_ratio"] = (
    state_daily["n_valid_stations"]
    / state_daily["n_station_records"].where(
        state_daily["n_station_records"] > 0
    )
)

assert not state_daily.duplicated(
    ["state", "weather_date"]
).any()

assert state_daily.loc[
    state_daily["n_valid_stations"].eq(0), "rainfall_mm"
].isna().all()

assert state_daily["rainfall_mm"].dropna().ge(0).all()

display(state_daily.head())

print("State-day rows:", len(state_daily))
print("States:", state_daily["state"].nunique())

,state,weather_date,rainfall_mm,n_valid_stations,n_station_records,station_record_completeness_ratio
0,AC,2016-01-01,2.766667,6,7,0.857143
1,AC,2016-01-02,7.1,6,7,0.857143
2,AC,2016-01-03,20.733333,6,7,0.857143
3,AC,2016-01-04,4.133333,6,7,0.857143
4,AC,2016-01-05,12.3,6,7,0.857143


State-day rows: 29592
States: 27


In [11]:
# Purpose: Review rainfall availability and station support by state.

state_weather_checks = (
    state_daily.groupby("state")
    .agg(
        total_days=("weather_date", "size"),
        valid_rain_days=("rainfall_mm", "count"),
        mean_valid_stations=("n_valid_stations", "mean"),
        min_valid_stations=("n_valid_stations", "min"),
    )
)

state_weather_checks["missing_rain_days"] = (
    state_weather_checks["total_days"]
    - state_weather_checks["valid_rain_days"]
)

state_weather_checks["missing_percent"] = (
    state_weather_checks["missing_rain_days"]
    / state_weather_checks["total_days"] * 100
).round(2)

display(
    state_weather_checks.sort_values(
        "missing_percent", ascending=False
    )
)

print(
    "State-days supported by only one station:",
    state_daily["n_valid_stations"].eq(1).sum(),
)

print(
    "State-days without valid rainfall:",
    state_daily["rainfall_mm"].isna().sum(),
)

,total_days,valid_rain_days,mean_valid_stations,min_valid_stations,missing_rain_days,missing_percent
state,,,,,,
RR,1096,982,1.222628,0,114,10.4
AP,1096,1037,1.732664,0,59,5.38
AC,1096,1080,3.849453,0,16,1.46
MT,1096,1095,24.311131,0,1,0.09
BA,1096,1096,38.321168,18,0,0.0
AM,1096,1096,13.418796,5,0,0.0
DF,1096,1096,3.482664,1,0,0.0
ES,1096,1096,9.877737,4,0,0.0
GO,1096,1096,20.232664,14,0,0.0


State-days supported by only one station: 1119
State-days without valid rainfall: 190


In [12]:
# Purpose: Create lagged rainfall features on a complete daily calendar.

weather_features = (
    state_daily
    .sort_values(["state", "weather_date"])
    .reset_index(drop=True)
    .copy()
)

# Verify one continuous daily calendar per state.
for state, group in weather_features.groupby("state"):
    expected_dates = pd.date_range(
        group["weather_date"].min(),
        group["weather_date"].max(),
        freq="D",
    )

    assert group["weather_date"].tolist() == expected_dates.tolist()

rain_groups = weather_features.groupby("state")["rainfall_mm"]

weather_features["rain_lag2_mm"] = rain_groups.shift(2)

weather_features["rain_3days_lag2_mm"] = (
    rain_groups.transform(
        lambda values: (
            values.shift(2)
            .rolling(window=3, min_periods=3)
            .sum()
        )
    )
)

# Retain station support for reviewing feature quality.
weather_features["valid_stations_lag2"] = (
    weather_features.groupby("state")["n_valid_stations"]
    .shift(2)
)

weather_features["min_valid_stations_3days_lag2"] = (
    weather_features.groupby("state")["n_valid_stations"]
    .transform(
        lambda values: (
            values.shift(2)
            .rolling(window=3, min_periods=3)
            .min()
        )
    )
)

display(weather_features[
    [
        "state",
        "weather_date",
        "rainfall_mm",
        "rain_lag2_mm",
        "rain_3days_lag2_mm",
    ]
].head(8))

,state,weather_date,rainfall_mm,rain_lag2_mm,rain_3days_lag2_mm
0,AC,2016-01-01,2.766667,<NA>,NaN
1,AC,2016-01-02,7.1,<NA>,NaN
2,AC,2016-01-03,20.733333,2.766667,NaN
3,AC,2016-01-04,4.133333,7.1,NaN
4,AC,2016-01-05,12.3,20.733333,30.600000
5,AC,2016-01-06,0.866667,4.133333,31.966667
6,AC,2016-01-07,4.166667,12.3,37.166667
7,AC,2016-01-08,11.133333,0.866667,17.300000


In [13]:
# Purpose: Verify lagged rainfall using an explicit date window.

example_state = "DF"
example_purchase_date = pd.Timestamp("2017-04-10")

source_start = example_purchase_date - pd.Timedelta(days=4)
source_end = example_purchase_date - pd.Timedelta(days=2)

source_rows = state_daily.loc[
    state_daily["state"].eq(example_state)
    & state_daily["weather_date"].between(source_start, source_end),
    ["weather_date", "rainfall_mm", "n_valid_stations"],
]

feature_row = weather_features.loc[
    weather_features["state"].eq(example_state)
    & weather_features["weather_date"].eq(example_purchase_date),
    ["state", "weather_date", "rain_lag2_mm", "rain_3days_lag2_mm"],
]

display(source_rows)
display(feature_row)

expected_total = source_rows["rainfall_mm"].sum(min_count=3)
actual_total = feature_row["rain_3days_lag2_mm"].iloc[0]

assert (
    pd.isna(expected_total) and pd.isna(actual_total)
) or (
    pd.notna(expected_total)
    and pd.notna(actual_total)
    and abs(expected_total - actual_total) < 1e-8
)

print("Rainfall window check passed.")

,weather_date,rainfall_mm,n_valid_stations
7037,2017-04-06,8.25,4
7038,2017-04-07,0.05,4
7039,2017-04-08,0.0,4


,state,weather_date,rain_lag2_mm,rain_3days_lag2_mm
7041,DF,2017-04-10,0.0,8.3


Rainfall window check passed.


In [14]:
# Purpose: Load order dates, customer states, and seller assignments.

DATA_DIR = PROJECT_ROOT / "data/processed"

orders = pd.read_csv(
    DATA_DIR / "research_data.csv",
    usecols=["order_id", "order_purchase_timestamp", "customer_state"],
    dtype={"order_id": "string", "customer_state": "string"},
)

items = pd.read_csv(
    DATA_DIR / "items_clean.csv",
    usecols=["order_id", "seller_id"],
    dtype="string",
)

sellers = pd.read_csv(
    DATA_DIR / "sellers_clean.csv",
    usecols=["seller_id", "seller_state"],
    dtype="string",
)

assert orders["order_id"].notna().all()
assert orders["order_id"].is_unique
assert sellers["seller_id"].notna().all()
assert sellers["seller_id"].is_unique
assert items[["order_id", "seller_id"]].notna().all().all()

orders["purchase_date"] = pd.to_datetime(
    orders["order_purchase_timestamp"],
    errors="raise",
).dt.normalize()

orders["customer_state"] = (
    orders["customer_state"].str.strip().str.upper()
)

sellers["seller_state"] = (
    sellers["seller_state"].str.strip().str.upper()
)

print("Orders:", len(orders))
print("Distinct order-seller pairs:", len(
    items.drop_duplicates(["order_id", "seller_id"])
))

Orders: 99441
Distinct order-seller pairs: 100010


In [15]:
# Purpose: Attach lagged rainfall at the customer's state.

weather_columns = [
    "rain_lag2_mm",
    "rain_3days_lag2_mm",
    "valid_stations_lag2",
    "min_valid_stations_3days_lag2",
]

customer_weather = orders[
    ["order_id", "customer_state", "purchase_date"]
].merge(
    weather_features[
        ["state", "weather_date"] + weather_columns
    ],
    left_on=["customer_state", "purchase_date"],
    right_on=["state", "weather_date"],
    how="left",
    validate="many_to_one",
)

customer_weather = customer_weather.rename(columns={
    column: f"customer_{column}"
    for column in weather_columns
})

customer_feature_columns = [
    f"customer_{column}" for column in weather_columns
]

order_weather = customer_weather[
    ["order_id"] + customer_feature_columns
].copy()

assert len(order_weather) == len(orders)
assert order_weather["order_id"].is_unique

display(order_weather.head())

display(
    order_weather[customer_feature_columns]
    .isna().sum()
    .to_frame("missing_rows")
)

,order_id,customer_rain_lag2_mm,customer_rain_3days_lag2_mm,customer_valid_stations_lag2,customer_min_valid_stations_3days_lag2
0,e481f51cbdc54678b7cc49136f2d6af7,17.133333,23.705114,33.0,32.0
1,53cdb2fc8bc7dce0b6741e2150273451,0.22439,4.957143,41.0,41.0
2,47770eb9100c2d0c44946d9cf07ec65d,0.0,0.000000,23.0,23.0
3,949d5b44dbf5de918fe9c16f97b45f8a,0.0,0.350000,7.0,7.0
4,ad21c59c0840e6cb83a9ceb5573f8159,6.539394,19.704100,33.0,33.0


,missing_rows
customer_rain_lag2_mm,8
customer_rain_3days_lag2_mm,9
customer_valid_stations_lag2,0
customer_min_valid_stations_3days_lag2,0


In [16]:
# Purpose: Attach rainfall to each distinct order-seller pair.

order_sellers = items.drop_duplicates(
    ["order_id", "seller_id"]
)

order_sellers = order_sellers.merge(
    sellers,
    on="seller_id",
    how="left",
    validate="many_to_one",
)

order_sellers = order_sellers.merge(
    orders[["order_id", "purchase_date"]],
    on="order_id",
    how="left",
    validate="many_to_one",
)

seller_weather = order_sellers.merge(
    weather_features[
        ["state", "weather_date"] + weather_columns
    ],
    left_on=["seller_state", "purchase_date"],
    right_on=["state", "weather_date"],
    how="left",
    validate="many_to_one",
)

assert len(seller_weather) == len(order_sellers)

seller_summary = (
    seller_weather.groupby("order_id")
    .agg(
        weather_seller_count=("seller_id", "size"),
        seller_rain_lag2_mean_mm=("rain_lag2_mm", "mean"),
        seller_rain_3days_lag2_mean_mm=(
            "rain_3days_lag2_mm", "mean"
        ),
        seller_rain_lag2_known_count=("rain_lag2_mm", "count"),
        seller_rain_3days_lag2_known_count=(
            "rain_3days_lag2_mm", "count"
        ),
    )
    .reset_index()
)

seller_summary["seller_rain_lag2_coverage_ratio"] = (
    seller_summary["seller_rain_lag2_known_count"]
    / seller_summary["weather_seller_count"]
)

seller_summary["seller_rain_3days_lag2_coverage_ratio"] = (
    seller_summary["seller_rain_3days_lag2_known_count"]
    / seller_summary["weather_seller_count"]
)

# Retain every original order, including orders without item records.
order_weather = (
    order_weather[["order_id"] + customer_feature_columns]
    .merge(
        seller_summary,
        on="order_id",
        how="left",
        validate="one_to_one",
    )
)

assert len(order_weather) == len(orders)
assert order_weather["order_id"].is_unique
assert order_weather["order_id"].equals(orders["order_id"])

display(order_weather.head())

display(
    order_weather.drop(columns="order_id")
    .isna().sum()
    .to_frame("missing_rows")
)

,order_id,customer_rain_lag2_mm,customer_rain_3days_lag2_mm,customer_valid_stations_lag2,customer_min_valid_stations_3days_lag2,weather_seller_count,seller_rain_lag2_mean_mm,seller_rain_3days_lag2_mean_mm,seller_rain_lag2_known_count,seller_rain_3days_lag2_known_count,seller_rain_lag2_coverage_ratio,seller_rain_3days_lag2_coverage_ratio
0,e481f51cbdc54678b7cc49136f2d6af7,17.133333,23.705114,33.0,32.0,1,17.133333,23.705114,1,1.0,1.0,1.0
1,53cdb2fc8bc7dce0b6741e2150273451,0.22439,4.957143,41.0,41.0,1,0.228571,0.308571,1,1.0,1.0,1.0
2,47770eb9100c2d0c44946d9cf07ec65d,0.0,0.000000,23.0,23.0,1,12.25,19.165152,1,1.0,1.0,1.0
3,949d5b44dbf5de918fe9c16f97b45f8a,0.0,0.350000,7.0,7.0,1,0.255738,3.213632,1,1.0,1.0,1.0
4,ad21c59c0840e6cb83a9ceb5573f8159,6.539394,19.704100,33.0,33.0,1,6.539394,19.704100,1,1.0,1.0,1.0


,missing_rows
customer_rain_lag2_mm,8
customer_rain_3days_lag2_mm,9
customer_valid_stations_lag2,0
customer_min_valid_stations_3days_lag2,0
weather_seller_count,775
seller_rain_lag2_mean_mm,775
seller_rain_3days_lag2_mean_mm,775
seller_rain_lag2_known_count,775
seller_rain_3days_lag2_known_count,775
seller_rain_lag2_coverage_ratio,775


In [19]:
# Purpose: Validate rainfall values and seller weather coverage.

rain_columns = [
    "customer_rain_lag2_mm",
    "customer_rain_3days_lag2_mm",
    "seller_rain_lag2_mean_mm",
    "seller_rain_3days_lag2_mean_mm",
]

assert len(order_weather) == len(orders)
assert order_weather["order_id"].is_unique
assert order_weather["order_id"].equals(orders["order_id"])

for column in rain_columns:
    values = order_weather[column].dropna()
    assert values.ge(0).all()
    assert not values.isin([float("inf"), -float("inf")]).any()

has_seller = order_weather["weather_seller_count"].notna()

coverage_checks = []

for window in ["lag2", "3days_lag2"]:
    coverage_column = f"seller_rain_{window}_coverage_ratio"
    known_column = f"seller_rain_{window}_known_count"
    rain_column = f"seller_rain_{window}_mean_mm"

    coverage = order_weather[coverage_column]

    assert coverage.dropna().between(0, 1).all()

    # A seller average exists exactly when at least one value is known.
   # Compare boolean values independently of their storage dtype.
has_average = (
    order_weather.loc[has_seller, rain_column]
    .notna()
    .astype("boolean")
)

has_known_values = (
    order_weather.loc[has_seller, known_column]
    .gt(0)
    .astype("boolean")
)

mismatches = (
    has_average.ne(has_known_values)
    | has_known_values.isna()
)

if mismatches.any():
    bad_indices = mismatches.index[mismatches]

    display(order_weather.loc[
        bad_indices,
        [
            "order_id",
            "weather_seller_count",
            rain_column,
            known_column,
            coverage_column,
        ],
    ].head(20))

    raise ValueError(
        f"{window}: inconsistent seller weather rows: "
        f"{int(mismatches.sum())}"
    )

    coverage_checks.append({
        "window": window,
        "orders_without_seller_records": int((~has_seller).sum()),
        "orders_with_no_weather": int(
            (has_seller & coverage.eq(0)).sum()
        ),
        "orders_with_partial_weather": int(
            (has_seller & coverage.gt(0) & coverage.lt(1)).sum()
        ),
        "orders_with_complete_weather": int(
            (has_seller & coverage.eq(1)).sum()
        ),
    })

display(pd.DataFrame(coverage_checks))

display(pd.DataFrame({
    "missing_rows": order_weather[rain_columns].isna().sum(),
    "missing_percent": (
        order_weather[rain_columns].isna().mean() * 100
    ).round(3),
}))

print("All order weather checks passed.")

""


,missing_rows,missing_percent
customer_rain_lag2_mm,8,0.008
customer_rain_3days_lag2_mm,9,0.009
seller_rain_lag2_mean_mm,775,0.779
seller_rain_3days_lag2_mean_mm,775,0.779


All order weather checks passed.


In [23]:
# Purpose: Save weather data, order features, and quality reports.

WEATHER_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

state_daily.to_csv(
    WEATHER_OUTPUT_DIR / "state_daily_weather.csv",
    index=False,
    date_format="%Y-%m-%d",
)

order_weather.to_csv(
    WEATHER_OUTPUT_DIR / "order_weather_features.csv",
    index=False,
)

quality_dir = PROJECT_ROOT / "outputs/tables/weather_quality"
quality_dir.mkdir(parents=True, exist_ok=True)

weather_file_audit.to_csv(
    quality_dir / "weather_file_audit.csv",
    index=False,
)

state_weather_checks.to_csv(
    quality_dir / "state_weather_checks.csv",
    index=True,
)

pd.DataFrame(coverage_checks).to_csv(
    quality_dir / "order_weather_coverage_checks.csv",
    index=False,
)

print("Saved weather data to:", WEATHER_OUTPUT_DIR)
print("State daily shape:", state_daily.shape)
print("Order weather shape:", order_weather.shape)
print("Saved quality reports to:", quality_dir)

Saved weather data to: c:\Users\LENOVO\Downloads\Group5_STAT3013.R11-main (1)\Group5_STAT3013.R11-main\data\processed
State daily shape: (29592, 6)
Order weather shape: (99441, 12)
Saved quality reports to: c:\Users\LENOVO\Downloads\Group5_STAT3013.R11-main (1)\Group5_STAT3013.R11-main\outputs\tables\weather_quality


In [22]:
# Purpose: Rebuild, validate, and export seller weather coverage checks.

has_seller = order_weather["weather_seller_count"].notna()
coverage_rows = []

for window in ["lag2", "3days_lag2"]:
    coverage_column = f"seller_rain_{window}_coverage_ratio"
    known_column = f"seller_rain_{window}_known_count"
    rain_column = f"seller_rain_{window}_mean_mm"

    coverage = order_weather[coverage_column]
    known = order_weather[known_column]
    seller_count = order_weather["weather_seller_count"]

    # Check coverage inputs for orders with seller records.
    assert seller_count.loc[has_seller].gt(0).all()
    assert known.loc[has_seller].notna().all()
    assert coverage.loc[has_seller].notna().all()
    assert coverage.loc[has_seller].between(0, 1).all()
    assert known.loc[has_seller].between(
        0, seller_count.loc[has_seller]
    ).all()

    # Compare values without requiring identical boolean dtypes.
    has_average = (
        order_weather.loc[has_seller, rain_column]
        .notna()
        .to_numpy(dtype=bool)
    )

    has_known_values = (
        known.loc[has_seller]
        .gt(0)
        .to_numpy(dtype=bool)
    )

    assert (has_average == has_known_values).all()

    row = {
        "window": window,
        "orders_without_seller_records": int((~has_seller).sum()),
        "orders_with_no_weather": int(
            (has_seller & coverage.eq(0)).sum()
        ),
        "orders_with_partial_weather": int(
            (has_seller & coverage.gt(0) & coverage.lt(1)).sum()
        ),
        "orders_with_complete_weather": int(
            (has_seller & coverage.eq(1)).sum()
        ),
    }

    assert sum(
        value for key, value in row.items() if key != "window"
    ) == len(order_weather)

    coverage_rows.append(row)

coverage_table = pd.DataFrame(coverage_rows)

# Keep compatibility with the earlier export cell.
coverage_checks = coverage_table.to_dict("records")

quality_dir = PROJECT_ROOT / "outputs/tables/weather_quality"
quality_dir.mkdir(parents=True, exist_ok=True)

output_path = quality_dir / "order_weather_coverage_checks.csv"
coverage_table.to_csv(output_path, index=False)

display(coverage_table)
print("Coverage checks passed.")
print("Saved:", output_path.resolve())

,window,orders_without_seller_records,orders_with_no_weather,orders_with_partial_weather,orders_with_complete_weather
0,lag2,775,0,0,98666
1,3days_lag2,775,0,0,98666


Coverage checks passed.
Saved: C:\Users\LENOVO\Downloads\Group5_STAT3013.R11-main (1)\Group5_STAT3013.R11-main\outputs\tables\weather_quality\order_weather_coverage_checks.csv
